## SIRS with Human Behaviour and Double Memory
The model here is similar to the previous one, but instead we choose:
$$
W(t) = a \tau e^{-a \tau}
$$
so that the resulting equations for the memory are:
$$
\begin{cases}
\dot{M_1} = a_1 (I - M_1) \\
\dot{M_2} = a_2 (M_1 - M_2)
\end{cases}
$$
I.e. this is the case in which people have two different *layers* of memory, one that is very close to the present and one other very aged.

At the same time we had to change the function for $\beta$, so that it reflects the changing:
$$
\beta(M) = \beta(M_1, M_2) = \frac{\beta_0}{(1 + c_1 M_1)(1 + c_2 M_2)}
$$

In [ ]:
import numpy as np
from matplotlib.pyplot import subplots
from plotly.subplots import make_subplots
from scipy.integrate import solve_ivp
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [ ]:
def SIRS_double (t, X, beta, gamma, mu, theta, a1, a2):
    I,R, M1, M2 = X

    # dS = (mu + theta) - S*(mu + theta) - beta(M1, M2)*I*S - theta*I
    dI = I*( beta(M1, M2)*(1-R-I) - (mu + gamma) )
    dR = gamma*I - (mu + theta)*R
    dM1 = a1 * (I - M1)
    dM2 = a2 * (M1 - M2)

    return [dI,dR, dM1, dM2]

In [ ]:
def SIRS_incidence_double(t, X, beta, gamma, mu, theta, a1, a2):
    S, I, M1, M2 = X

    dS = (mu + theta)*(1-S) - beta(M1, M2)*S*I - theta*I
    dI = I*( beta(M1, M2)*S - (mu + gamma) )
    dM1 = a1 * (beta(M1, M2)*S*I - M1)
    dM2 = a2 * (M1 - M2)

    return [dS, dI, dM1, dM2]

In [ ]:
def beta_double (beta0, d1, d2):
    def beta_double_inside(M1, M2):
        return beta0 / (1+d1*M1) / (1+d2*M2)
    return beta_double_inside

In [ ]:
# parameters

gamma = 4
mu = 1/(80*12)
theta = 1/60

a1 = 1/8                  # Short term memory  characteristic time        3 months
a2 = 1/120                 # Long term memory characteristic time          5 years
beta_0 = 3.9*(mu+gamma)

d1 = 5                      # Short term memory weight
d2 = 2                      # Long term memory weight
beta = beta_double(beta_0, d1, d2)

# Initial conditions
S0 = 0.999                  # Initial suspicious population
I0 = 0.001                  # Initial infected population
R0 = 0.
M10 = 0.                     # Initial short memory rate
M20 = 0.                   # Initial long term memory rate
# X0 = [I0, R0, M10, M20]
X0 = [S0, I0, M10, M20]

# t = 100 month
t_span = (0, 26)

In [ ]:
# Solve the system of ODEs
# solution = solve_ivp(SIRS_double, t_span, X0, args=(beta, gamma, mu, theta, a1, a2), dense_output=True)
solution = solve_ivp(SIRS_incidence_double, t_span, X0, args=(beta, gamma, mu, theta, a1, a2), dense_output=True)

# Time points for which to get the solution
t = np.linspace(t_span[0], t_span[1], 100000)
sol = solution.sol(t)
# Calculating some specifics of the model

# Some specifics of the model
mean_m1 = np.mean(sol[2])
mean_m2 = np.mean(sol[3])
mean_R_0 = beta(mean_m1, mean_m2) / (mu + gamma)
beta_e = beta(sol[2][-1], sol[3][-1])
R_0_ee = beta_e / (mu + gamma)

# Endemic equilibrium
S_e = float(1 / R_0_ee)
I_e = float((1 - 1 / R_0_ee) * (mu + theta) / (mu + gamma + theta))
R_e = float((1 - 1 / R_0_ee) * gamma / (mu + gamma + theta))
# print parameters
print(
    f"Parameters: \nGamma: {gamma}\t\tMu: {mu}\t\tTheta: {round(theta, 5)}\t\tR0 at equilibrium: {round(R_0_ee, 5)}\n")
print(f"Specifics: \nEndemic Equilibrium: {round(S_e, 10), round(I_e, 5), round(R_e, 5)}")

In [ ]:
fig = px.line(x=t, y=[0] * len(sol[0]), title=f' Behavioural SIRS with Double Memory     -      a₁:{round(a1, 3)}, a₂:{round(a2, 3)}', labels={'x': 'Time', 'y': 'Population'},
              color_discrete_sequence=['rgba(0, 0, 0, 0)'])
#b
fig.add_scatter(x=t, y=1 - sol[0] - sol[1], mode='lines', name='Recovered', line=dict(color='blue'))
fig.add_scatter(x=t, y=sol[0], mode='lines', name='Suspicious', line=dict(color='red'))
fig.add_scatter(x=t, y=sol[1], mode='lines', name='Infectious', line=dict(color='green'))
#
fig.show()

In [ ]:
M1_e = sol[2][-1]
M2_e = sol[3][-1]

beta1 = d1*beta_0 / (1+d2*M2_e) / (1+d1*M1_e)**2
beta2 = d2*beta_0 / (1+d1*M1_e) / (1+d2*M2_e)**2

h = beta_e*I_e
k = beta_e*S_e
x = h+mu + theta
y = k+mu + theta
w = k*(mu + theta)
z = h*(mu + theta)
sigma1 = - beta1*S_e*I_e
sigma2 = - beta2*S_e*I_e

q3 =a1*(sigma1 + 1) + a2 + x
q2 = a1*a2*(sigma1+sigma2 + 1) + a1*(sigma1*y + x) + a2*x + z
q1 = a1*a2*(y*(sigma1 + sigma2)+x) + a1*(sigma1*w+z) + a2*z
q0 = a1*a2*(z+w*(sigma1+sigma2))

In [ ]:
(q1 / q3)**2 - q2*q1 / q3 + q0

In [ ]:
fig = px.scatter(x=t[:40000], y=[0] * len(sol[2][:40000]), title='I(t) and the memories M1(t) and M2(t)',
                 labels={'x': 'Months', 'y': 'Value'}, color_discrete_sequence=['rgba(0, 0, 0, 0)'])
fig.add_scatter(x=t[:40000], y=sol[1][:40000], name='I(t)', line=dict(color='red'))
fig.add_scatter(x=t[:40000], y=sol[2][:40000], name='M1(t)', line=dict(color='blue'))
fig.add_scatter(x=t[:40000], y=sol[3][:40000], name='M2(t)', line=dict(color='rgb(0, 125, 255)'))

fig.show()

In [ ]:
betas_double = [beta(m1, m2) for m1, m2 in zip(sol[2], sol[3])]
one_betas_double = [beta_0 - i for i in betas_double]

In [ ]:
fig = px.scatter(x=t[:70000], y=[0] * len(sol[2][:70000]), title='β₀-β(t)',
                 color_discrete_sequence=['rgba(0, 0, 0, 0)'])
fig.add_scatter(x=t[:70000], y=sol[2][:70000], name='M1', line=dict(color='blue'))
fig.add_scatter(x=t[:70000], y=sol[3][:70000], name='M2', line=dict(color='rgb(0, 125, 255)'))
fig.add_scatter(x=t[:70000], y=one_betas_double[:70000], name='β₀-β(t)', line=dict(color='aquamarine'))

fig.show()

## Routh-Hurwitz Function plot
$$q_1 ^ 2 - q_1q_2q_3 + q_0q_3 ^ 2 = 0$$


In [ ]:
def beta_derivative(beta_0, d1, d2):
    def beta_derivative_inside(M1, M2, idx):
        res = - beta_0 * d2 / ((1 + d1 * M1) * (1 + d2 * M2) ** 2) if idx == 1 else - beta_0 * d1 / (
                    (1 + d2 * M2) * (1 + d1 * M1) ** 2)
        return res

    return beta_derivative_inside

In [ ]:
beta_d = beta_derivative(beta_0, d1, 2)
beta_ds_1 = [beta_d(M1, M2, 0) for M1, M2 in zip(sol[2], sol[3])]
beta_ds_2 = [beta_d(M1, M2, 1) for M1, M2 in zip(sol[2], sol[3])]

db1_e = beta_ds_1[-1]
db2_e = beta_ds_2[-1]

In [ ]:
c3 = beta_e * I_e + mu + theta
c2 = beta_e * I_e * (mu + gamma + theta)
c1 = -db1_e * I_e * S_e
c0 = -db2_e * I_e * S_e
q3 = a1 + a2 + c3
q2 = a1 * a2 + a1 * (c1 + c3) + a2 * c3 + c2
q1 = a1 * a2 * (c0 + c1 + c3) + a1 * (c2 + c1 * (mu + theta)) + a2 * c2
q0 = a1 * a2 * (c2 * (c0 + c1) * (mu + theta))

In [ ]:
(q1 / q3) ** 2 - q2 * q1 / q3 + q0

### Main Loop

In [ ]:
import os
print(os.getcwd())

In [ ]:
rhs = []

a1s = [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 3, 5, 10]
a2s = [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 3, 5, 10]



# parameters

gamma = 1/7
mu = 1/50/365.25
theta = 1/365.25

beta_0 = 5*(mu+gamma)

d1 = 0.2                      # Short term memory weight
d2 = 0.5                     # Long term memory weight
beta = beta_double(beta_0, d1, d2)

# Initial conditions
S0 = 0.999                  # Initial suspicious population
I0 = 0.001                  # Initial infected population
R0 = 0
M10 = 0.                     # Initial short memory rate
M20 = 0.                   # Initial long term memory rate
X0 = [I0,R0, M10, M20]

# t = 100 month
t_span = (0, 1200)


first = 0

for a1 in a1s:
    for a2 in a2s:
        # Solve the system of ODEs
        solution = solve_ivp(SIRS_double, t_span, X0, args=(beta, gamma, mu, theta, a1, a2), dense_output=True)

        # Time points for which to get the solution
        t = np.linspace(t_span[0], t_span[1], 100000)
        sol = solution.sol(t)

        # Calculating some specifics of the model
        mean_m1 = np.mean(sol[2])
        mean_m2 = np.mean(sol[3])
        mean_R_0 = beta(mean_m1, mean_m2) / (mu + gamma)
        beta_e = beta(sol[2][-1], sol[3][-1])
        R_0_ee = beta_e / (mu + gamma)

        # Endemic equilibrium
        S_e = float(1 / R_0_ee)
        I_e = float((1 - 1 / R_0_ee) * (mu + theta) / (mu + gamma + theta))
        R_e = float((1 - 1 / R_0_ee) * gamma / (mu + gamma + theta))

        c3 = beta_e * I_e + mu + theta
        c2 = beta_e * I_e * (mu + gamma + theta)
        c1 = -db1_e * I_e * S_e
        c0 = -db2_e * I_e * S_e

        q3 = a1 + a2 + c3
        q2 = a1 * a2 + a1 * (c1 + c3) + a2 * c3 + c2
        q1 = a1 * a2 * (c0 + c1 + c3) + a1 * (c2 + c1 * (mu + theta)) + a2 * c2
        q0 = a1 * a2 * (c2 * (c0 + c1) * (mu + theta))

        rh = (q1 / q3) ** 2 - q2 * q1 / q3 + q0

        # print parameters
        # print(f'\n\na1: {a1}, a2: {a2}',
        #       f"Parameters: \nGamma: {gamma}\t\tMu: {mu}\t\tTheta: {round(theta, 5)}\t\tR0 at equilibrium: {round(R_0_ee, 5)}\n",
        #       f"Specifics: \nEndemic Equilibrium: {round(S_e, 10), round(I_e, 5), round(R_e, 5)},"
        #       f"q0: {q0}, q1: {q1}, q2: {q2}, q3: {q3}, RH= {rh}")

        rhs.append([a1, a2, rh])

        if first == 100:
            fig = px.line(x=t, y=[0] * len(sol[0]), title='SIRS - Human Behaviour', labels={'x': 'Time', 'y': 'Population'},
              color_discrete_sequence=['rgba(0, 0, 0, 0)'])
            #
            fig.add_scatter(x=t, y=sol[0], mode='lines', name='Suspicious', line=dict(color='blue'))
            fig.add_scatter(x=t, y=sol[1], mode='lines', name='Infected', line=dict(color='red'))
            fig.add_scatter(x=t, y=1 - sol[0] - sol[1], mode='lines', name='Recovered', line=dict(color='green'))

            # save in file:
            fig.write_html('grafico2.html')

        first += 1

In [ ]:
from plotly.express import scatter_3d

fig = px.scatter_3d(
    rhs,
    x=[i[0] for i in rhs],
    y=[i[1] for i in rhs],
    z=[i[2] for i in rhs],
    title='Routh-Hurwitz Function',
    labels={'x': 'a1', 'y': 'a2', 'z': 'Routh-Hurwitz Function'},
)
fig.update_layout(
    scene=dict(
        xaxis_title='a1',
        yaxis_title='a2',
        zaxis_title='Routh-Hurwitz Function',
    ),
    margin=dict(l=0, r=0, b=0, t=0),
)
# smaller dots
fig.update_traces(marker=dict(size=2))
fig.show()

In [ ]:
from plotly import graph_objects as go

fig = go.Figure(data=[go.Mesh3d(
    x=[x[0] for x in rhs],
    y=[y[1] for y in rhs],
    z=[z[2] for z in rhs],
    color='lightblue',
    opacity=0.50
)])
fig.show()

## Simulations Chap. 4.5

In [ ]:
a1s = [1/16, 1/8, 1/4, 1/2, 1, 2, 30]
a2s = [1/240, 1/120, 1/60, 1/30, 1/24, 1/6, 1/3]
limit = 12000

In [ ]:
a1 = 30
a2 = 1/3
theta = 1/60

fig = make_subplots(rows=7, cols=2, subplot_titles=(f'Behavioural SIRS   -    a₁:{round(a1, 3)}, a₂:{round(a2, 3)}', f'Phase plan for (I, R)'))
fig.update_layout(width=1000, height=1800, template='plotly_white')

for i, (a1, a2) in enumerate(zip(a1s, a2s)):
    model = solve_ivp(SIRS_double, t_span, X0, args=(beta, gamma, mu, theta, a1, a2), dense_output=True)
    sol = model.sol(t)

    # Some specifics of the model
    mean_m1 = np.mean(sol[2])
    mean_m2 = np.mean(sol[3])
    mean_R_0 = beta(mean_m1, mean_m2) / (mu + gamma)
    beta_e = beta(sol[2][-1], sol[3][-1])
    R_0_ee = beta_e / (mu + gamma)

    # Endemic equilibrium
    S_e = float(1 / R_0_ee)
    I_e = float((1 - 1 / R_0_ee) * (mu + theta) / (mu + gamma + theta))
    R_e = float((1 - 1 / R_0_ee) * gamma / (mu + gamma + theta))
    # print parameters
    print(
        f"Parameters: \nGamma: {gamma}\t\tMu: {round(mu, 5)}\t\tTheta: {round(theta, 5)}\t\tR0 at equilibrium: {round(R_0_ee, 5)}\n")
    print(f"Specifics: \nEndemic Equilibrium: {round(S_e, 10), round(I_e, 5), round(R_e, 5)}\n\n")


    Is = [sol[0][i]/I_e for i in range(len(sol[0][:limit]))]
    Rs = [sol[1][i]/R_e for i in range(len(sol[1][:limit]))]



    # Create subplot figure


    # ----------- LEFT subplot traces (with legendgroup 'group1') ----------------
    fig.add_trace(go.Scatter(
        x=t[:limit], y=(1 - sol[1] - sol[0])[:limit],
        mode='lines', name='Susceptibles',
        line=dict(color='rgba(0, 0, 255, 0.3)'),
        legendgroup='group1', showlegend=True,
    ), row=i+1, col=1)

    fig.add_trace(go.Scatter(
        x=t[:limit], y=sol[0][:limit],
        mode='lines', name='Infected',
        line=dict(color='red'),
        legendgroup='group1', showlegend=True,
    ), row=i+1, col=1)

    fig.add_trace(go.Scatter(
        x=t[:limit], y=sol[1][:limit],
        mode='lines', name='Recovered',
        line=dict(color='rgba(0, 255, 0, 0.6)'),
        legendgroup='group1', showlegend=True,
    ), row=i+1, col=1)

    # ----------- RIGHT subplot trace (with legendgroup 'group2') ----------------

    fig.add_trace(go.Scatter(
        x=Is[:limit], y=Rs[:limit],
        mode='markers', name='Phase trajectory',
        marker=dict(color=t[:limit], size=5),
        legendgroup='group2', showlegend=True
    ), row=i+1, col=2)

    # ----------- DUMMY trace for second "legend" ----------------
    #fig.add_trace(go.Scatter(
    #     x=[None], y=[None],
    #     mode='markers', name='Phase trajectory',
    #     legendgroup='group2', showlegend=True
    # ), row=1, col=2)

    # ----------- Layout with simulated stacked legends ----------------

    fig.update_layout(
        template='plotly_white',
        legend=dict(
            x=1.05,          # right of the plot
            y=1,             # top of the first subplot
            xanchor='left',
            yanchor='top',
            traceorder='normal',
            itemsizing='constant'
        ),
        margin=dict(r=150)  # ensure enough space on the right
    )

    # Manual offset of second group: Reduce y position for next set (workaround)
    # This only works if second legend item appears right after the first ones
    # because there's only one legend box, so ordering matters!

fig.show()





## Simulations Cap. 4.6

In [ ]:
theta = 1/60

fig = make_subplots(rows=7, cols=2, subplot_titles=(f'Behavioural SIRS   -    a₁:{round(a1, 3)}, a₂:{round(a2, 3)}', f'Phase plan for (S, I)'))
fig.update_layout(width=1000, height=1800, template='plotly_white')

for i, (a1, a2) in enumerate(zip(a1s, a2s)):
    a1 = 0.125
    a2 = 0.008
    model = solve_ivp(SIRS_incidence_double, t_span, X0, args=(beta, gamma, mu, theta, a1, a2), dense_output=True)
    sol = model.sol(t)

    # Some specifics of the model
    mean_m1 = np.mean(sol[2])
    mean_m2 = np.mean(sol[3])
    mean_R_0 = beta(mean_m1, mean_m2) / (mu + gamma)
    beta_e = beta(sol[2][-1], sol[3][-1])
    R_0_ee = beta_e / (mu + gamma)

    # Endemic equilibrium
    S_e = float(1 / R_0_ee)
    I_e = float((1 - 1 / R_0_ee) * (mu + theta) / (mu + gamma + theta))
    R_e = float((1 - 1 / R_0_ee) * gamma / (mu + gamma + theta))
    # print parameters
    print(
        f"Parameters: \nGamma: {gamma}\t\tMu: {round(mu, 5)}\t\tTheta: {round(theta, 5)}\t\tR0 at equilibrium: {round(R_0_ee, 5)}\na1: {round(a1, 3)}\t\t a2: {round(a2, 3)}")
    print(f"Specifics: \nEndemic Equilibrium: {round(S_e, 10), round(I_e, 5), round(R_e, 5)}\n\n")


    Ss = [sol[0][i]/S_e for i in range(len(sol[0][:limit]))]
    Is = [sol[1][i]/I_e for i in range(len(sol[1][:limit]))]



    # Create subplot figure


    # ----------- LEFT subplot traces (with legendgroup 'group1') ----------------
    fig.add_trace(go.Scatter(
        x=t[:limit], y=sol[0][:limit],
        mode='lines', name='Susceptibles',
        line=dict(color='rgba(0, 0, 255, 0.3)'),
        legendgroup='group1', showlegend=True,
    ), row=i+1, col=1)

    fig.add_trace(go.Scatter(
        x=t[:limit], y=sol[1][:limit],
        mode='lines', name='Infected',
        line=dict(color='red'),
        legendgroup='group1', showlegend=True,
    ), row=i+1, col=1)

    fig.add_trace(go.Scatter(
        x=t[:limit], y=(1 - sol[0]-sol[1])[:limit],
        mode='lines', name='Recovered',
        line=dict(color='rgba(0, 255, 0, 0.6)'),
        legendgroup='group1', showlegend=True,
    ), row=i+1, col=1)

    # ----------- RIGHT subplot trace (with legendgroup 'group2') ----------------

    fig.add_trace(go.Scatter(
        x=Ss[:limit], y=Is[:limit],
        mode='markers', name='Phase trajectory',
        marker=dict(color=t[:limit], size=5),
        legendgroup='group2', showlegend=True
    ), row=i+1, col=2)


    # ----------- Layout with simulated stacked legends ----------------

    fig.update_layout(
        template='plotly_white',
        legend=dict(
            x=1.05,          # right of the plot
            y=1,             # top of the first subplot
            xanchor='left',
            yanchor='top',
            traceorder='normal',
            itemsizing='constant'
        ),
        margin=dict(r=150)  # ensure enough space on the right
    )

    # Manual offset of second group: Reduce y position for next set (workaround)
    # This only works if second legend item appears right after the first ones
    # because there's only one legend box, so ordering matters!

fig.show()



